In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10
from torchvision import transforms

from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

rootdir = "/opt/img/effdl-cifar10/"
working_dir = "/users/local/myworkingdir"

os.makedirs(working_dir, exist_ok=True)

train_embeddings_path = os.path.join(working_dir, "dinov2_train_embeddings.pt")
test_embeddings_path = os.path.join(working_dir, "dinov2_test_embeddings.pt")
linear_probe_path = os.path.join(working_dir, "linear_probe.pth")
mlp_head_path = os.path.join(working_dir, "mlp_head.pth")

In [ ]:
torch.hub.set_dir(working_dir)

backbone = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
)

backbone.eval()
backbone.to(device)

for param in backbone.parameters():
    param.requires_grad = False

print("DINOv2 loaded and frozen.")

In [ ]:
transform_dino = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [ ]:
train_dataset = CIFAR10(
    root=rootdir,
    train=True,
    download=True,
    transform=transform_dino
)

test_dataset = CIFAR10(
    root=rootdir,
    train=False,
    download=True,
    transform=transform_dino
)

class_names = train_dataset.classes

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))
print("Classes:", class_names)

In [ ]:
extraction_batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=extraction_batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=extraction_batch_size,
    shuffle=False
)

In [ ]:
def extract_embeddings(loader, model, device):
    model.eval()

    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            outputs = model.forward_features(images)

            features = outputs["x_norm_clstoken"]

            all_features.append(features.cpu())
            all_labels.append(labels)

    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_features, all_labels

In [ ]:
train_features, train_labels = extract_embeddings(
    train_loader,
    backbone,
    device
)

test_features, test_labels = extract_embeddings(
    test_loader,
    backbone,
    device
)

print("Train features shape:", train_features.shape)
print("Train labels shape:", train_labels.shape)

print("Test features shape:", test_features.shape)
print("Test labels shape:", test_labels.shape)

torch.save(
    {
        "features": train_features,
        "labels": train_labels
    },
    train_embeddings_path
)

torch.save(
    {
        "features": test_features,
        "labels": test_labels
    },
    test_embeddings_path
)

print("Embeddings saved.")

In [ ]:
train_features.shape = (50000, 384)
test_features.shape = (10000, 384)

In [ ]:
del backbone

if device.type == "cuda":
    torch.cuda.empty_cache()

print("Backbone removed from memory.")

In [ ]:
train_data = torch.load(train_embeddings_path)
test_data = torch.load(test_embeddings_path)

X_train = train_data["features"]
y_train = train_data["labels"]

X_test = test_data["features"]
y_test = test_data["labels"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
classifier_batch_size = 128

train_tensor_dataset = TensorDataset(X_train, y_train)
test_tensor_dataset = TensorDataset(X_test, y_test)

train_embedding_loader = DataLoader(
    train_tensor_dataset,
    batch_size=classifier_batch_size,
    shuffle=True
)

test_embedding_loader = DataLoader(
    test_tensor_dataset,
    batch_size=classifier_batch_size,
    shuffle=False
)

In [ ]:
def train_one_epoch_classifier(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, labels in loader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(features)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * features.size(0)

        _, predicted = torch.max(outputs, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


def evaluate_classifier(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * features.size(0)

            _, predicted = torch.max(outputs, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc, all_labels, all_predictions

In [ ]:
linear_probe = nn.Linear(384, 10).to(device)

criterion = nn.CrossEntropyLoss()

optimizer_linear = optim.AdamW(
    linear_probe.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

num_epochs_linear = 20

In [ ]:
linear_train_losses = []
linear_test_losses = []

linear_train_accuracies = []
linear_test_accuracies = []

best_linear_acc = 0.0

for epoch in range(num_epochs_linear):
    train_loss, train_acc = train_one_epoch_classifier(
        linear_probe,
        train_embedding_loader,
        criterion,
        optimizer_linear,
        device
    )

    test_loss, test_acc, _, _ = evaluate_classifier(
        linear_probe,
        test_embedding_loader,
        criterion,
        device
    )

    linear_train_losses.append(train_loss)
    linear_test_losses.append(test_loss)

    linear_train_accuracies.append(train_acc)
    linear_test_accuracies.append(test_acc)

    print(
        f"Epoch [{epoch+1}/{num_epochs_linear}] "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc:.4f} | "
        f"Test loss: {test_loss:.4f} | "
        f"Test acc: {test_acc:.4f}"
    )

    if test_acc > best_linear_acc:
        best_linear_acc = test_acc

        torch.save(
            {
                "net": linear_probe.state_dict(),
                "best_test_acc": best_linear_acc
            },
            linear_probe_path
        )

        print("Best linear probe saved.")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(linear_train_losses, label="Train loss")
plt.plot(linear_test_losses, label="Test loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Linear probe - Loss")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(linear_train_accuracies, label="Train accuracy")
plt.plot(linear_test_accuracies, label="Test accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Linear probe - Accuracy")
plt.legend()
plt.grid()
plt.show()

In [ ]:
class MLPHead(nn.Module):
    def __init__(self, input_dim=384, num_classes=10):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
mlp_head = MLPHead().to(device)

optimizer_mlp = optim.AdamW(
    mlp_head.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

num_epochs_mlp = 20

In [ ]:
mlp_train_losses = []
mlp_test_losses = []

mlp_train_accuracies = []
mlp_test_accuracies = []

best_mlp_acc = 0.0

for epoch in range(num_epochs_mlp):
    train_loss, train_acc = train_one_epoch_classifier(
        mlp_head,
        train_embedding_loader,
        criterion,
        optimizer_mlp,
        device
    )

    test_loss, test_acc, _, _ = evaluate_classifier(
        mlp_head,
        test_embedding_loader,
        criterion,
        device
    )

    mlp_train_losses.append(train_loss)
    mlp_test_losses.append(test_loss)

    mlp_train_accuracies.append(train_acc)
    mlp_test_accuracies.append(test_acc)

    print(
        f"Epoch [{epoch+1}/{num_epochs_mlp}] "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc:.4f} | "
        f"Test loss: {test_loss:.4f} | "
        f"Test acc: {test_acc:.4f}"
    )

    if test_acc > best_mlp_acc:
        best_mlp_acc = test_acc

        torch.save(
            {
                "net": mlp_head.state_dict(),
                "best_test_acc": best_mlp_acc
            },
            mlp_head_path
        )

        print("Best MLP head saved.")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(mlp_train_losses, label="Train loss")
plt.plot(mlp_test_losses, label="Test loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("MLP head - Loss")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(mlp_train_accuracies, label="Train accuracy")
plt.plot(mlp_test_accuracies, label="Test accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("MLP head - Accuracy")
plt.legend()
plt.grid()
plt.show()

In [ ]:
linear_checkpoint = torch.load(linear_probe_path, map_location=device)
mlp_checkpoint = torch.load(mlp_head_path, map_location=device)

best_linear_probe = nn.Linear(384, 10).to(device)
best_linear_probe.load_state_dict(linear_checkpoint["net"])
best_linear_probe.eval()

best_mlp_head = MLPHead().to(device)
best_mlp_head.load_state_dict(mlp_checkpoint["net"])
best_mlp_head.eval()

print("Best linear test accuracy:", linear_checkpoint["best_test_acc"])
print("Best MLP test accuracy:", mlp_checkpoint["best_test_acc"])

In [ ]:
linear_loss, linear_acc, y_true_linear, y_pred_linear = evaluate_classifier(
    best_linear_probe,
    test_embedding_loader,
    criterion,
    device
)

mlp_loss, mlp_acc, y_true_mlp, y_pred_mlp = evaluate_classifier(
    best_mlp_head,
    test_embedding_loader,
    criterion,
    device
)

print("Linear probe accuracy:", linear_acc)
print("MLP head accuracy:", mlp_acc)

print("\nLinear probe classification report:")
print(classification_report(
    y_true_linear,
    y_pred_linear,
    target_names=class_names
))

print("\nMLP head classification report:")
print(classification_report(
    y_true_mlp,
    y_pred_mlp,
    target_names=class_names
))

In [ ]:
cm = confusion_matrix(y_true_mlp, y_pred_mlp)

plt.figure(figsize=(9, 8))
plt.imshow(cm)
plt.title("Confusion matrix - DINOv2 + MLP head")
plt.colorbar()

plt.xticks(range(10), class_names, rotation=45)
plt.yticks(range(10), class_names)

plt.xlabel("Predicted label")
plt.ylabel("True label")

plt.show()

In [ ]:
num_tsne_samples = 2000
rng = np.random.RandomState(seed=42)

indices = rng.choice(
    len(X_train),
    size=num_tsne_samples,
    replace=False
)

X_tsne = X_train[indices].numpy()
y_tsne = y_train[indices].numpy()

print("t-SNE input shape:", X_tsne.shape)

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=42
)

X_tsne_2d = tsne.fit_transform(X_tsne)

print("t-SNE output shape:", X_tsne_2d.shape)

In [ ]:
plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    X_tsne_2d[:, 0],
    X_tsne_2d[:, 1],
    c=y_tsne,
    s=8,
    cmap="tab10"
)

plt.colorbar(scatter, ticks=range(10), label="Class label")
plt.title("t-SNE visualization of DINOv2 embeddings")
plt.xlabel("t-SNE dimension 1")
plt.ylabel("t-SNE dimension 2")
plt.grid()
plt.show()

In [ ]:
resnet_from_scratch_acc = None  # remplacer par ton résultat du TP 2

results = {
    "Classifier": [
        "ResNet from scratch",
        "DINOv2 + Linear probe",
        "DINOv2 + MLP head"
    ],
    "Test Accuracy": [
        resnet_from_scratch_acc,
        linear_acc,
        mlp_acc
    ]
}

for classifier, acc in zip(results["Classifier"], results["Test Accuracy"]):
    print(classifier, ":", acc)